# Equal-trial-budget nested spatial comparison
Parameter selection uses inner spatial folds; reported scores come from outer
spatial folds. Prior feature selection used this data, so this remains development
evidence. Full run: 600 fits. Existing trained models and original holdout unchanged.


In [ ]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
from IPython.display import display
ROOT = Path(r"C:/digitalTwin")
RUN_SMOKE = False
RUN_FULL = False
# Optional: exact interrupted output folder; use matching smoke/full settings.
RESUME = None
print("Kernel:", sys.executable)
assert (ROOT / "scripts/27_nested_spatial_comparison.py").is_file()


In [ ]:
if RUN_SMOKE and RUN_FULL:
    raise ValueError("Choose smoke OR full, not both.")
if RUN_SMOKE or RUN_FULL:
    args = [sys.executable, "-u", str(ROOT / "scripts/27_nested_spatial_comparison.py"),
            "--root", str(ROOT), "--threads", "2"]
    if RUN_SMOKE:
        args += ["--smoke"]
    if RESUME is not None:
        args += ["--resume", str(RESUME)]
    subprocess.run(args, cwd=ROOT, check=True)
else:
    print("Training disabled. Review completed full results below.")


In [ ]:
runs = sorted(p for p in (ROOT / "models/experiments").glob("nested_spatial_*")
              if (p / "COMPLETED.json").is_file()
              and json.loads((p / "COMPLETED.json").read_text())["status"] == "NESTED_DEVELOPMENT_COMPLETE")
if not runs:
    print("No completed full nested run yet. Smoke results are excluded.")
else:
    run = runs[-1]
    print(run)
    summary = pd.read_csv(run / "summary.csv")
    display(summary.pivot(index="model", columns="variant", values="rmse"))
    display(summary[["model","variant","r2","hot_rmse","cold_rmse","bias"]])
    display(pd.read_csv(run / "paired_comparison.csv"))
    display(pd.read_csv(run / "outer_results.csv")[[
        "outer_fold","model","variant","selected_trial","selected_inner_rmse","rmse"]])


## Interpretation
Compare matched outer-fold results and report hot/cold trade-offs. Equal numbers
of trials do not equal globally optimized models or equal computing cost.
Do not use these results to claim first-ever novelty, fresh validation or causal
cooling. The feature design was informed by this data historically.
No trained estimator is saved or promoted by this script.
